### 02 - Feature Engineering

This notebook creates the first modeling dataset using NFL schedule and play-by-play data from 2010-2025.

This workflow ultimately creates the baseline feature dataset with all of the base features needed to predict games. This notebook calculates historical averages available before each game begins, turns plays into team-game statistics, along with combining home and away features so that there is one row per game which includes home and away stats.   

The target is `home_win`: 1 for a home win and 0 for an away win. Tied games are excluded from from the dataset. Both regular-season and playoff games are included in the data.

In [1]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
SEASONS = list(range(2010, 2026))
TEST_SEASON = 2025
pbp_columns = ['game_id', 'posteam', 'defteam', 'epa', 'success', 'pass', 'rush', 'qb_kneel', 'qb_spike', 'two_point_attempt']
pbp = nfl.load_pbp(SEASONS).select(pbp_columns).to_pandas()
schedule = nfl.load_schedules(SEASONS).to_pandas()

### Selecting Plays for Team Efficiency Metrics

The analysis retains passing and rushing plays where teams, epa, and success are included. Quarterback kneels, spikes, and two-point attempts are excluded so that only ordinary offensive plays are used in features.



In [3]:
scrimmage_plays = pbp.loc[
    pbp["posteam"].notna()
    & pbp["defteam"].notna()
    & pbp["epa"].notna()
    & pbp["success"].notna()
    & (pbp["pass"].eq(1) | pbp["rush"].eq(1))
    & pbp["qb_kneel"].fillna(0).ne(1)
    & pbp["qb_spike"].fillna(0).ne(1)
    & pbp["two_point_attempt"].fillna(0).ne(1)
].copy()

### Aggregating Offensive and Defensive Performance

For each team game, I calculate offensive EPA per play, offensive success rate, passing EPA, and rushing EPA to represent offensive performance.
Defensive performance is summarized using the calculated EPA allowed and success rate allowed. 

Higher values for offensive features generally indicate strong offense performance, while lower defensive EPA and success rate allowed indicate stronge defense performance.

These are statistics from completed games. They become prediction inputs only after being converted into pregame averages based on historical data.

In [4]:
offense = (
    scrimmage_plays
    .groupby(['game_id', 'posteam'])
    .agg(
        off_epa = ('epa', 'mean'),
        off_success_rate = ('success', 'mean')
    )
    .reset_index()
)
pass_df = (
    scrimmage_plays.loc[scrimmage_plays["pass"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(pass_epa = ('epa', 'mean'))
    .reset_index()
)
rush_df = (
    scrimmage_plays.loc[scrimmage_plays["rush"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(rush_epa = ('epa', 'mean'))
    .reset_index()
)

In [5]:
offense_df = (
    offense
    .merge(pass_df, on=['game_id', 'posteam'], how='left')
    .merge(rush_df, on=['game_id', 'posteam'], how='left')
    .rename(columns={'posteam': 'team'})
)
offense_df.head()

,game_id,team,off_epa,off_success_rate,pass_epa,rush_epa
0,2010_01_ARI_STL,ARI,-0.136437,0.352941,0.092355,-0.614819
1,2010_01_ARI_STL,LA,-0.195926,0.358025,-0.235471,-0.102009
2,2010_01_ATL_PIT,ATL,-0.120100,0.445946,-0.074867,-0.208756
3,2010_01_ATL_PIT,PIT,-0.028287,0.366667,-0.080712,0.027754
4,2010_01_BAL_NYJ,BAL,-0.114943,0.395062,0.095338,-0.391312


In [6]:
defense_df = (
    scrimmage_plays
    .groupby(['game_id', 'defteam'])
    .agg(
        def_epa_allowed = ('epa', 'mean'),
        def_success_rate_allowed = ('success', 'mean'),
    )
    .reset_index()
    .rename(columns={'defteam': 'team'})
)
defense_df.head()

,game_id,team,def_epa_allowed,def_success_rate_allowed
0,2010_01_ARI_STL,ARI,-0.195926,0.358025
1,2010_01_ARI_STL,LA,-0.136437,0.352941
2,2010_01_ATL_PIT,ATL,-0.028287,0.366667
3,2010_01_ATL_PIT,PIT,-0.120100,0.445946
4,2010_01_BAL_NYJ,BAL,-0.411004,0.311111


### Preparing Game Identifiers, Outcomes, and Context

Historical team abbreviations are standardized so that schedule records can correctly join with play-by-play summaries. 

The binary target records whether the home team won. Ties receive a NaN value and are removed later on.

Rest days and neutral-site status describe the context of the game and how rested a team is. For neutral site games, both home and away teams are still assigned, but the home-field indicator is set to zero.

In [7]:
schedule_df = schedule.loc[
    schedule['home_score'].notna()
    & schedule['away_score'].notna(),
    ['game_id', 'season', 'week', 'gameday', 'game_type', 'home_team', 'away_team', 'home_score', 'away_score', 'home_rest', 'away_rest', 'location'],
].copy()

In [8]:
TEAM_ABBR_MAP = {
    'OAK': 'LV',
    'SD': 'LAC',
    'STL': 'LA',
}

schedule_df['home_team'] = schedule_df['home_team'].replace(TEAM_ABBR_MAP)
schedule_df['away_team'] = schedule_df['away_team'].replace(TEAM_ABBR_MAP)

In [9]:
schedule_df['home_win'] = np.where(
    schedule_df['home_score'].eq(schedule_df['away_score']), np.nan,
    schedule_df['home_score'].gt(schedule_df['away_score']).astype(int)
)

In [10]:
schedule_df['neutral_site'] = (schedule_df['location'].astype(str).str.lower().eq('neutral')).astype(int)

### Building One Row per Team per Game

Each schedule record is expanded into two rows, one from the home teams perspective and one from the away team's. These rows will be later remerged after calculations are complete.

This structure makes it simpler to calculate each teams historical performances. Offensive and defensive summaries are joined using `game_id` and `team`, and records are ordered chronologically. 

For the historical team win percentages, a tie contributes 0.5. This is solely for this calculation, and ties will be dropped later on.

In [11]:
home_rows = schedule_df[['season', 'week', 'gameday', 'game_id', 'game_type', 'neutral_site']].copy()

home_rows['team'] = schedule_df['home_team']
home_rows['opponent'] = schedule_df['away_team']
home_rows['points_scored'] = schedule_df['home_score']
home_rows['points_allowed'] = schedule_df['away_score']
home_rows['rest_days'] = schedule_df['home_rest']
home_rows['is_home'] = 1

away_rows = schedule_df[['season', 'week', 'gameday', 'game_id', 'game_type', 'neutral_site']].copy()

away_rows['team'] = schedule_df['away_team']
away_rows['opponent'] = schedule_df['home_team']
away_rows['points_scored'] = schedule_df['away_score']
away_rows['points_allowed'] = schedule_df['home_score']
away_rows['rest_days'] = schedule_df['away_rest']
away_rows['is_home'] = 0

In [12]:
def add_team_result(df):
    df = df.copy()

    df['win'] = np.select(
        [
            df['points_scored'].gt(df['points_allowed']),
            df['points_scored'].lt(df['points_allowed'])
        ],
        [1.0, 0.0],
        default=0.5,
    )
    df['home_field'] = (
    df['is_home'].eq(1) 
    & df['neutral_site'].eq(0)
    ).astype(int)

    return df

home_rows = add_team_result(home_rows)
away_rows = add_team_result(away_rows)

In [13]:
schedule_team_games_df = pd.concat([home_rows, away_rows], ignore_index=True)

In [14]:
team_games_df = (
    schedule_team_games_df
    .merge(
        offense_df, 
        on=['game_id', 'team'], 
        how='left',
    )
    .merge(
        defense_df, 
        on=['game_id', 'team'], 
        how='left',
    )    
)
team_games_df = (
    team_games_df
    .sort_values(['team', 'gameday', 'game_id'])
    .reset_index(drop=True)
)

In [15]:
team_games_df = team_games_df[[
    'season',
    'week',
    'gameday',  
    'game_id',
    'game_type',
    'team',
    'opponent',
    'is_home',
    'neutral_site',
    'home_field',
    'rest_days',
    'off_epa',
    'off_success_rate',
    'pass_epa',
    'rush_epa',
    'def_epa_allowed',
    'def_success_rate_allowed',
    'points_scored',
    'points_allowed',
    'win',
]]

In [16]:
team_games_df.loc[
    team_games_df["game_id"] == "2025_01_BAL_BUF"
]

,season,week,gameday,game_id,game_type,team,opponent,is_home,neutral_site,home_field,rest_days,off_epa,off_success_rate,pass_epa,rush_epa,def_epa_allowed,def_success_rate_allowed,points_scored,points_allowed,win
798,2025,1,2025-09-07,2025_01_BAL_BUF,REG,BAL,BUF,0,0,0,7,0.394435,0.509804,0.530099,0.282997,0.302612,0.512821,40,41,0.0
1072,2025,1,2025-09-07,2025_01_BAL_BUF,REG,BUF,BAL,1,0,1,7,0.302612,0.512821,0.457560,-0.046021,0.394435,0.509804,41,40,1.0


### Converting Completed-Game Statistics into Pregame Features

Pregame features use expanding averages within each team and season. Applying `shift(1)` before the average excludes the current games statistics from leaking into pregame features.

Each completed game contributes equally to these averages. The first game of each season for each team has no current season history. So for week 1, each team has missing pregame averages. The checks below inspect this behavior and confirm that the values are missing for week 1.

In [17]:
team_games_df['pregame_off_epa'] = (
    team_games_df
    .groupby(['season', 'team'])['off_epa']
    .transform(lambda x: x.shift(1).expanding().mean())
)

team_games_df = (
    team_games_df
    .sort_values(['team', 'gameday', 'game_id'])
    .reset_index(drop=True)
)

In [18]:
pregame_feature_map = {
    'off_epa': 'pregame_off_epa',
    'off_success_rate': 'pregame_off_success_rate',
    'pass_epa': 'pregame_pass_epa',
    'rush_epa': 'pregame_rush_epa',
    'def_epa_allowed': 'pregame_def_epa_allowed',
    'def_success_rate_allowed': 'pregame_def_success_rate_allowed',
    'points_scored': 'pregame_ppg',
    'points_allowed': 'pregame_ppg_allowed',
    'win': 'pregame_win_pct',
}


In [19]:
for raw_column, pregame_column in pregame_feature_map.items():
    team_games_df[pregame_column] = (
        team_games_df
        .groupby(['season', 'team'], sort=False)[raw_column]
        .transform(lambda x: x.shift(1).expanding(min_periods=1).mean())
    )

In [20]:
team_games_df['prior_games'] = (
    team_games_df
    .groupby(['season', 'team'], sort=False)
    .cumcount()
)

In [21]:
team_games_df.loc[
    (team_games_df['season'].eq(TEST_SEASON))
    & (team_games_df['team'].eq('NE')),
    ['week', 'gameday', 'opponent', 'off_epa', 'pregame_off_epa', 'points_scored', 'pregame_ppg', 'win', 'pregame_win_pct','prior_games']
]

,week,gameday,opponent,off_epa,pregame_off_epa,points_scored,pregame_ppg,win,pregame_win_pct,prior_games
5972,1,2025-09-07,LV,-0.045775,NaN,13,NaN,0.0,NaN,0
5973,2,2025-09-14,MIA,0.275018,-0.045775,33,13.000000,1.0,0.000000,1
5974,3,2025-09-21,PIT,-0.109990,0.114622,14,23.000000,0.0,0.500000,2
5975,4,2025-09-28,CAR,0.354956,0.039751,42,20.000000,1.0,0.333333,3
5976,5,2025-10-05,BUF,0.065326,0.118552,23,25.500000,1.0,0.500000,4
5977,6,2025-10-12,NO,0.138688,0.107907,25,25.000000,1.0,0.600000,5
5978,7,2025-10-19,TEN,0.267477,0.113037,31,25.000000,1.0,0.666667,6
5979,8,2025-10-26,CLE,0.093407,0.135100,32,25.857143,1.0,0.714286,7
5980,9,2025-11-02,ATL,0.013419,0.129888,24,26.625000,1.0,0.750000,8
5981,10,2025-11-09,TB,0.193619,0.116947,28,26.333333,1.0,0.777778,9


In [22]:
pregame_columns = list(pregame_feature_map.values())

first_team_games = (
    team_games_df
    .groupby(["season", "team"], sort=False)
    .head(1)
)

assert first_team_games[pregame_columns].isna().all().all()

### Creating the Matchup Dataset

Home-team and away-team pregame features are attached to the schedule to produce one row per game. In the row for each game, features are given either the `home` or `away` prefix to identify which team the feature is for.  

V1 requires that both teams have played at least one earlier game that season. Because of this, in the V1 dataset, all week 1 games are removed. 

Tied games are removed from the model dataset. Final scores and current-game performance statistics are not included as predicting features in the dataset.

In [23]:
team_feature_columns = [
    'game_id', 'team', 'prior_games', *pregame_columns,
]
pregame_team_features_df = team_games_df[team_feature_columns].copy()

In [24]:
home_features_df = pregame_team_features_df.rename(
    columns={
        'team': 'home_team',
        'prior_games': 'home_prior_games',
        **{
            column: f"home_{column}"
            for column in pregame_columns
        },
    }
)

away_features_df = pregame_team_features_df.rename(
    columns={
        'team': 'away_team',
        'prior_games': 'away_prior_games',
        **{
            column: f"away_{column}"
            for column in pregame_columns
        },
    }
)

In [25]:
game_columns = [
    'game_id', 'season', 'week', 'gameday', 'game_type', 'home_team', 'away_team', 'neutral_site', 'home_rest', 'away_rest', 'home_win'
]
games = schedule_df[game_columns].copy()

In [26]:
games = games.rename(
    columns={
        'home_rest': 'home_rest_days',
        'away_rest': 'away_rest_days',
    }
)
games['home_field'] = 1 - games['neutral_site']

In [27]:
model_df = (
    games
    .merge(home_features_df, on=['game_id', 'home_team'], how='left')
    .merge(away_features_df, on=['game_id', 'away_team'], how='left')
)

In [28]:
model_df = model_df.loc[
    model_df['home_prior_games'].ge(1)
    & model_df['away_prior_games'].ge(1)
].copy()

In [29]:
model_df = model_df.dropna(subset=['home_win']).copy()
model_df['home_win'] = model_df['home_win'].astype(int)

model_df = model_df.drop(columns=['home_prior_games', 'away_prior_games'])

model_df = model_df.sort_values(['gameday', 'game_id']).reset_index(drop=True)

In [30]:
print(model_df.shape)
print(model_df.columns)
model_df.head()

(4096, 30)
Index(['game_id', 'season', 'week', 'gameday', 'game_type', 'home_team',
       'away_team', 'neutral_site', 'home_rest_days', 'away_rest_days',
       'home_win', 'home_field', 'home_pregame_off_epa',
       'home_pregame_off_success_rate', 'home_pregame_pass_epa',
       'home_pregame_rush_epa', 'home_pregame_def_epa_allowed',
       'home_pregame_def_success_rate_allowed', 'home_pregame_ppg',
       'home_pregame_ppg_allowed', 'home_pregame_win_pct',
       'away_pregame_off_epa', 'away_pregame_off_success_rate',
       'away_pregame_pass_epa', 'away_pregame_rush_epa',
       'away_pregame_def_epa_allowed', 'away_pregame_def_success_rate_allowed',
       'away_pregame_ppg', 'away_pregame_ppg_allowed', 'away_pregame_win_pct'],
      dtype='str')


,game_id,season,week,gameday,game_type,home_team,away_team,neutral_site,home_rest_days,away_rest_days,...,home_pregame_win_pct,away_pregame_off_epa,away_pregame_off_success_rate,away_pregame_pass_epa,away_pregame_rush_epa,away_pregame_def_epa_allowed,away_pregame_def_success_rate_allowed,away_pregame_ppg,away_pregame_ppg_allowed,away_pregame_win_pct
0,2010_02_ARI_ATL,2010,2,2010-09-19,REG,ATL,ARI,0,7,7,...,0.0,-0.136437,0.352941,0.092355,-0.614819,-0.195926,0.358025,17.0,13.0,1.0
1,2010_02_BAL_CIN,2010,2,2010-09-19,REG,CIN,BAL,0,7,6,...,0.0,-0.114943,0.395062,0.095338,-0.391312,-0.411004,0.311111,10.0,9.0,1.0
2,2010_02_BUF_GB,2010,2,2010-09-19,REG,GB,BUF,0,7,7,...,1.0,-0.231829,0.309091,-0.221807,-0.258552,-0.009411,0.418919,10.0,15.0,0.0
3,2010_02_CHI_DAL,2010,2,2010-09-19,REG,DAL,CHI,0,7,7,...,0.0,-0.151665,0.447368,-0.030265,-0.371984,-0.300751,0.321429,19.0,14.0,1.0
4,2010_02_HOU_WAS,2010,2,2010-09-19,REG,WAS,HOU,0,7,7,...,1.0,0.260859,0.610169,-0.032805,0.411455,0.174313,0.500000,34.0,24.0,1.0


### Export and V1 Limitations

The completed dataset for V1 is saved as `data/processed/model_dataset_v1.csv` for baseline modeling in Notebook 3. V1 uses season-to-date averages, which provide a simple starting point but do not explicitly represent recent form, quarterback identity, opponent strength, or previous-season information. It solely relies on current season history, which limits it from predicting week 1 games.

These limitations are resolved in the V2 dataset developed in Notebook 4.

In [31]:
project_root = Path.cwd().resolve()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

assert (project_root / "notebooks").exists(), (
    f"Could not identify project root from: {project_root}"
)

output_directory = project_root / "data" / "processed"
output_directory.mkdir(parents=True, exist_ok=True)

output_path = output_directory / "model_dataset_v1.csv"

model_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8",
)

print(f"Saved {len(model_df):,} rows to:")
print(output_path)

Saved 4,096 rows to:
C:\Coding Projects\NFL-2026-Machine-Learning-Project\data\processed\model_dataset_v1.csv
